# Module 2 — Steering + MMLU capability check

**v2 §3 capability gate.** For each emotion (desperation, calm, sad, angry), sweep α ∈ {0.025, 0.05, 0.075, 0.1} and measure MMLU accuracy under steering. Report the largest α where the MMLU drop from baseline is ≤ 1pt.

**Prerequisite:** M1 must have completed and dropped vectors into `outputs/m1_vectors/{emotion}.npy`.

**Pipeline:**
1. Load Gemma-2-9B-IT (same A100 box as M1).
2. Estimate mean residual norm at the extraction layer using neutral corpus (this is what α multiplies).
3. Run unsteered MMLU baseline.
4. Sweep α × 4 emotions = 16 steered runs; record accuracy each time.
5. Apply v2 capability gate and report per-emotion max α.

**Cost:** ~4 GPU-hr on A100 (16 sweeps × 1140 MMLU prompts × 1 forward pass each + 1 baseline + 1 calibration). ~$10 on Vast/Lambda spot, ~50 compute units on Colab Pro+.

**Output:** `outputs/m2/mmlu_by_alpha_emotion.csv`, `outputs/m2/decision.txt`.

## Cell 1 — env setup

In [ ]:
from google.colab import userdata
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
# ANTHROPIC_API_KEY is not needed for M2 — kept here so the cell mirrors the sanity / M1 setup
try:
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    pass
!git clone https://github.com/BraydenFeng/Algoverse.git || (cd Algoverse && git pull)
%cd Algoverse

!pip install -q -r requirements.txt

## Cell 2 — load Gemma-2-9B-IT and verify M1 vectors are on disk

In [ ]:
import sys
sys.path.insert(0, '.')

from pathlib import Path
import numpy as np

from src.lib.config import load_config
from src.lib.model_load import load_gemma
from src.steering import load_emotion_vector

cfg = load_config()
emotions = cfg['extraction']['emotions']
layer = cfg['models']['primary']['extraction_layer']
alphas = cfg['steering']['alpha_sweep']
drop_tol = cfg['steering']['mmlu_drop_tolerance']

# verify M1 ran — fail fast with a clear message if not
vectors = {e: load_emotion_vector(e) for e in emotions}
for e, v in vectors.items():
    print(f'{e}: shape={v.shape}, ||v||={np.linalg.norm(v):.4f}')

model, tokenizer = load_gemma(variant='primary')
print(f'loaded {model.config._name_or_path}, n_layers={model.config.num_hidden_layers}, d_model={model.config.hidden_size}, steering layer={layer}')

## Cell 3 — estimate mean residual norm at the steering layer

Anthropic convention: α is a fraction of `||h||` at the target layer. We measure this on the neutral corpus (committed in `data/stories/neutral/`) so the calibration is reproducible. One pass through the 20 neutral passages, ~1 min.

In [ ]:
from src.steering import estimate_residual_norm

data_dir = Path(cfg['paths']['data_dir'])
neutral_texts = [p.read_text(encoding='utf-8') for p in sorted((data_dir / 'stories' / 'neutral').glob('*.txt'))]
print(f'calibrating on {len(neutral_texts)} neutral passages')

norm_scale = estimate_residual_norm(
    model, tokenizer, layer=layer,
    calibration_texts=neutral_texts,
    token_skip=cfg['extraction']['token_skip'],
)
print(f'mean ||h|| at layer {layer} = {norm_scale:.2f}')
print(f'α=0.025 → steering magnitude ≈ {0.025 * norm_scale:.2f}')
print(f'α=0.10  → steering magnitude ≈ {0.10 * norm_scale:.2f}')

## Cell 4 — unsteered MMLU baseline

Stratified 20 prompts × 57 subjects = 1140 prompts. Log-prob scoring over (A, B, C, D). ~15 min on A100. This number is the reference the steered sweeps are compared against.

In [ ]:
from src.mmlu_eval import run_mmlu

baseline = run_mmlu(model, tokenizer)
print(f'baseline MMLU: {baseline.accuracy:.4f} ({baseline.n_correct}/{baseline.n_total})')

## Cell 5 — sweep α × emotions

16 runs total. Each ~15 min → ~4 hr wall clock. The hook is registered before MMLU and removed after; nothing else changes between runs, so per-run cost is the MMLU pass itself.

In [ ]:
from src.steering import make_steering_hook_factory
import json
import pandas as pd

outputs_dir = Path(cfg['paths']['outputs_dir']) / 'm2'
outputs_dir.mkdir(parents=True, exist_ok=True)

rows = []
for emotion in emotions:
    for alpha in alphas:
        print(f'\n=== {emotion} @ α={alpha} ===')
        hook_factory = make_steering_hook_factory(
            vector=vectors[emotion],
            layer=layer,
            alpha=alpha,
            norm_scale=norm_scale,
        )
        result = run_mmlu(model, tokenizer, pre_forward_hook=hook_factory)
        drop = baseline.accuracy - result.accuracy
        rows.append({
            'emotion': emotion,
            'alpha': alpha,
            'accuracy': result.accuracy,
            'n_correct': result.n_correct,
            'n_total': result.n_total,
            'drop_from_baseline': drop,
            'within_tolerance': drop <= drop_tol,
        })
        # checkpoint the full per-row prediction CSV per (emotion, alpha) so a crash
        # mid-sweep doesn't lose hours of compute
        result.per_row.to_csv(outputs_dir / f'predictions_{emotion}_a{alpha}.csv', index=False)
        print(f'acc={result.accuracy:.4f}, drop={drop:+.4f}, within {drop_tol:.2f} tol: {drop <= drop_tol}')

df = pd.DataFrame(rows)
df.to_csv(outputs_dir / 'mmlu_by_alpha_emotion.csv', index=False)
df

## Cell 6 — apply capability gate, report per-emotion max α

For each emotion, the v2 result is the largest α from the sweep that satisfies `drop ≤ drop_tol`. If no α in the sweep satisfies the gate, the emotion has no usable steering strength under this protocol and that's a finding.

In [ ]:
lines = ['Module 2 — Steering + MMLU capability check', '=' * 50, '']
lines.append(f'baseline MMLU accuracy: {baseline.accuracy:.4f} ({baseline.n_correct}/{baseline.n_total})')
lines.append(f'drop tolerance: {drop_tol:.2f}')
lines.append(f'norm scale ||h||@L{layer}: {norm_scale:.2f}')
lines.append('')
lines.append('per-emotion max usable α:')

for emotion in emotions:
    sub = df[(df['emotion'] == emotion) & (df['within_tolerance'])].sort_values('alpha', ascending=False)
    if len(sub):
        winner = sub.iloc[0]
        lines.append(f'  {emotion:13s}  α={winner["alpha"]:.4f}  acc={winner["accuracy"]:.4f}  drop={winner["drop_from_baseline"]:+.4f}')
    else:
        lines.append(f'  {emotion:13s}  NO usable α in sweep — all α exceeded drop tolerance')

lines.append('')
lines.append('full grid:')
lines.append(df.to_string(index=False))

decision = '\n'.join(lines)
print(decision)
(outputs_dir / 'decision.txt').write_text(decision, encoding='utf-8')

## Outputs

- `outputs/m2/mmlu_by_alpha_emotion.csv` — grid of (emotion, α) → accuracy, drop, gate-pass.
- `outputs/m2/predictions_{emotion}_a{alpha}.csv` — per-prompt MMLU predictions for each sweep cell (for slicing by subject if any emotion shows weird per-subject behavior).
- `outputs/m2/decision.txt` — human-readable summary + per-emotion winner.

## Push artifacts to HF Hub

```python
from huggingface_hub import HfApi
HfApi().upload_folder(
    folder_path='outputs/m2',
    repo_id=cfg['paths']['hf_artifact_repo'],
    repo_type='dataset',
    path_in_repo='m2_results',
)
```

## Next

The per-emotion winning α from this notebook is the α used in M3 (steered FaithEval). If desperation has a usable α (typical case), proceed to `m3_faitheval.ipynb`. If desperation has NO usable α, the steering hypothesis fails the capability gate and M3 should be reconsidered before running.

In [ ]:
from huggingface_hub import HfApi
HfApi().upload_folder(
    folder_path='outputs/m2',
    repo_id=cfg['paths']['hf_artifact_repo'],
    repo_type='dataset',
    path_in_repo='m2_results',
)